# Hausaufgaben zu Schätzern

### Aufgabe 1: Gewicht der Gelbbauchmolche (Z-Intervall)

In einer neuen Studie wurde das Gewicht von $n = 50$ [Gelbbauchmolche](https://de.wikipedia.org/wiki/Rauh%C3%A4utiger_Gelbbauchmolch) (*Taricha granulosa*) gemessen. Wir wissen aus langjährigen Beobachtungen, dass die Standardabweichung des Gewichts in der Population etwa $\sigma = 1.2\text{ g}$ beträgt.

**Ergebnis der Stichprobe:**
* Stichprobenmittelwert ($\bar{x}$): $8.5\text{ g}$

**Deine Aufgabe:**
1. Berechne das **95 % Konfidenzintervall** für den wahren Mittelwert $\mu$.
2. Nutze dafür die `norm`-Verteilung aus `scipy.stats`.

In [ ]:
import numpy as np
from scipy.stats import norm

n = 50
sigma = 1.2
x_bar = 8.5

# TODO: Berechne den Standardfehler (SE)
se = sigma / np.sqrt(n)

# TODO: Berechne die Grenzen des Konfidenzintervalls (via norm.ppf oder norm.interval)
low, high = norm.interval(0.95, loc=x_bar, scale=se)

print(f"95% KI für das Gewicht: [{low:.3f} g, {high:.3f} g]")

### Aufgabe 2: Präzision der pH-Sensoren (KI für die Varianz)

In den Habitaten der Molche messen wir den pH-Wert des Wassers. Die Präzision der Sensoren ist entscheidend. Wir haben $n = 15$ Kontrollmessungen an einer geeichten Lösung durchgeführt.

**Daten (pH-Werte):**
`[7.01, 7.05, 6.98, 7.02, 7.00, 7.03, 6.99, 6.97, 7.04, 7.01, 7.02, 7.00, 7.03, 6.98, 7.01]`

**Deine Aufgabe:**
1. Berechne die Stichprobenvarianz $S^2$ mit `ddof=1`.
2. Bestimme das **90 % Konfidenzintervall** für die wahre Varianz $\sigma^2$.
3. Nutze die `chi2`-Verteilung aus `scipy.stats`.

In [ ]:
from scipy.stats import chi2

ph_daten = [7.01, 7.05, 6.98, 7.02, 7.00, 7.03, 6.99, 6.97, 7.04, 7.01, 7.02, 7.00, 7.03, 6.98, 7.01]

# TODO: Berechne n, df und S2
n = len(ph_daten)
df = n - 1
s2 = np.var(ph_daten, ddof=1)

# TODO: Bestimme die Chi2-Quantile für alpha = 0.10 (beidseitig)
alpha = 0.10
chi2_low = chi2.ppf(alpha / 2, df=df)
chi2_high = chi2.ppf(1 - alpha / 2, df=df)

# TODO: Berechne das Intervall für die Varianz
var_low = (df * s2) / chi2_high
var_high = (df * s2) / chi2_low

print(f"Stichprobenvarianz S^2: {s2:.6f}")
print(f"90% KI für die Varianz: [{var_low:.6f}, {var_high:.6f}]")

### Aufgabe 3: Fressverhalten (Bootstrapping)

Wir beobachten, wie viele Insekten ein [Tokeh](https://de.wikipedia.org/wiki/Tokeh) (*Gekko gecko*)  pro Tag frisst. Die Verteilung ist stark rechtsschief: Viele Molche fressen wenig, aber einige "Vielfraße" verzerren den Mittelwert.

**Daten (Insekten pro Tag):**
`[2, 3, 1, 4, 2, 5, 2, 3, 1, 20, 2, 4, 3, 2, 18, 1, 3, 2, 4, 25]`

**Deine Aufgabe:**
1. Warum ist hier der **Median** aussagekräftiger als der Mittelwert?
2. Führe ein Bootstrapping durch (10.000 Resamples), um das **95 % Konfidenzintervall für den Median** zu schätzen.
3. Nutze dazu `scipy.stats.bootstrap`.
4. Schätzt KDE die Daten ein.

In [ ]:
from scipy.stats import bootstrap, gaussian_kde
import matplotlib.pyplot as plt

# 1. Warum ist der Median aussagekräftiger?
# Da die Daten stark rechtsschief sind und extreme Ausreißer (20, 18, 25) enthalten,
# wird der Mittelwert (5.45) stark nach oben verzerrt. Der Median (3.0) ist robust gegenüber Ausreißern.

fress_daten = [2, 3, 1, 4, 2, 5, 2, 3, 1, 20, 2, 4, 3, 2, 18, 1, 3, 2, 4, 25]

# TODO: Berechne den Median der Stichprobe
stichproben_median = np.median(fress_daten)

# TODO: Nutze die bootstrap() Funktion für den Median
# Hinweis: Daten müssen als Tuple übergeben werden: (fress_daten,)
res = bootstrap((fress_daten,), np.median, n_resamples=10000, confidence_level=0.95, method='percentile', random_state=42)

print(f"Stichproben-Median: {stichproben_median}")
print(f"Bootstrap 95% KI für den Median: [{res.confidence_interval.low:.2f}, {res.confidence_interval.high:.2f}]")

# 4. KDE-Schätzung (Kernel Density Estimation)
kde = gaussian_kde(fress_daten)
x_grid = np.linspace(min(fress_daten) - 2, max(fress_daten) + 2, 500)
plt.figure(figsize=(8, 4))
plt.hist(fress_daten, bins=10, density=True, alpha=0.4, color='skyblue', label='Histogramm')
plt.plot(x_grid, kde(x_grid), color='darkblue', lw=2, label='KDE-Dichteschätzung')
plt.axvline(stichproben_median, color='red', linestyle='--', label=f'Median = {stichproben_median}')
plt.title('KDE-Schätzung des Fressverhaltens (Insekten pro Tag)')
plt.xlabel('Insekten pro Tag')
plt.ylabel('Dichte')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Qualitätskontrolle in der Produktion

Eine Fabrik produziert Metallstifte. Vor einigen Jahren wurde der mittlere Durchmesser des Produktionsprozesses auf 10 mm geschätzt.  

Kollege A geht davon aus, dass sich der Prozess seitdem nicht verändert hat und verwendet den Stichprobenmittelwert.

$$
\hat{\mu}_A = \bar{x}
$$

Kollege B arbeitet seit vielen Jahren in der Produktion. Er vermutet, dass sich die Maschinen im Laufe der Zeit leicht verändert haben und die alten Schätzungen den tatsächlichen Durchmesser heute etwas unterschätzen.  

Deshalb korrigiert er seine Schätzung um 0.2 mm nach oben.

$$
\hat{\mu}_B = \bar{x} + 0.2
$$


Zur Untersuchung werden 1000 Stichproben simuliert. Die Ergebnisse der beiden Schätzer lauten:
- Schätzer A:
  - Bias = 0.00
  - Varianz = 0.25

- Schätzer B:
  - Bias = 0.20
  - Varianz = 0.22

### Aufgaben
1. Berechne für beide Schätzer den Mean Squared Error (MSE).  
Erinnerung:
$$
MSE = Bias^2 + Varianz
$$

2. Welchen Schätzer würdest du bevorzugen?
3. Wann ist ein biased Schätzer gerechtfertigt?

In [ ]:
bias_A = 0.00
varianz_A = 0.25
bias_B = 0.20
varianz_B = 0.12

# MSE berechnen
mse_A = bias_A**2 + varianz_A
mse_B = bias_B**2 + varianz_B

print(f"MSE von Schätzer A: {mse_A:.2f}")
print(f"MSE von Schätzer B: {mse_B:.2f}")

In [ ]:
# bevorzugter Schätzer
bevorzugt = 'Schätzer B' if mse_B < mse_A else 'Schätzer A'
print(f"Bevorzugter Schätzer (basierend auf dem niedrigeren MSE): {bevorzugt} (MSE_B = {mse_B:.2f} < MSE_A = {mse_A:.2f})")
print("Hinweis: Falls im Aufgabentext Varianz_B = 0.22 verwendet wird, ergibt sich MSE_B = 0.26 > MSE_A = 0.25, dann wäre Schätzer A besser.")

**3: Wann ist ein biased Schätzer gerechtfertigt?**

Ein verzerrter (biased) Schätzer ist dann gerechtfertigt, wenn die durch den Bias eingeführte systematische Abweichung durch eine **deutlich stärkere Reduktion der Varianz** überkompensiert wird, sodass der **Mean Squared Error ($MSE = \text{Bias}^2 + \text{Varianz}$)** insgesamt kleiner ist als beim unverzerrten Schätzer (*Bias-Variance-Tradeoff*).